# Expérimentations préliminaires — 1. Régression logistique

Régression logistique **standard** (maximum de vraisemblance, sans pénalité ni
hyperparamètre) : $P(\text{victoire locale}) = \sigma(\beta_0 + \boldsymbol\beta^\top \mathbf{z})$.

Pipeline : imputation (médiane) → standardisation → **ACP (critère de Kaiser, λ > 1)** →
régression logistique. Pour chaque pli, tout le pipeline est ajusté sur les saisons
d'entraînement, puis évalué sur la saison testée. Rien n'est ajusté sur la saison testée.

**Pourquoi l'ACP ?** Les 1 060 variables sont très redondantes (voir
`00_donnees_protocole`). L'ACP retire cette colinéarité, qui rend les coefficients
instables, et réduit fortement le nombre de paramètres à estimer.

**Combien de composantes ?** Critère de Kaiser : on garde les composantes dont la valeur
propre est supérieure à 1. Les variables étant standardisées (variance 1 chacune), une
composante retenue résume plus de variance qu'une variable d'origine. Le nombre de
composantes est déterminé sur l'entraînement de chaque pli.

**Protocole commun** (voir `src/nhl_forecast/features/equipes.py` et
`src/nhl_forecast/evaluation/preliminaire.py`) :

- saisons régulières **2009-10 à 2025-26**, statistiques d'**équipes** seulement, **toutes les
  colonnes** de `t_summary`, `t_percentages`, `t_summaryshooting`, `t_outshootoutshotby` et
  MoneyPuck (5 situations de jeu) ; seuls les identifiants, les colonnes vides ou constantes et
  les doublons exacts sont retirés, automatiquement ;
- validation **progressive** (*walk-forward*, fenêtre croissante) : chaque pli teste **une
  saison** et s'entraîne sur toutes les saisons antérieures ;
  - **développement** : plis 1 à 8, saisons testées 2016-17 à 2023-24 ;
  - **évaluation finale** : plis 9 et 10, saisons 2024-25 et 2025-26, réservées (à ne pas
    utiliser pour les choix de modélisation) ;
- hyperparamètres choisis sur les 20 % les plus récents des matchs d'entraînement du pli ;
- métriques : log loss, score de Brier, accuracy (seuil 0,5), résumées séparément pour chaque
  phase.

Écart avec le protocole proposé (initialisation 2005-06 à 2007-08, développement à partir de
2008-09) : MoneyPuck ne commence qu'en 2008-09, et `t_percentages` / `t_summaryshooting` en
2009-10. Les saisons 2005-06 à 2008-09 sont donc exclues de ce jeu de statistiques d'équipes ;
elles pourront servir à l'initialisation des modèles qui n'utilisent que les résultats (Elo).

Avec `SAUVER = True`, les tableaux sont enregistrés en CSV (séparateur « ; ») dans
`reports/map6009/tables/` (préfixe `exp_`) et les prédictions dans
`data/processed/experimentation/` (lues par `04_comparaison.ipynb`).

L'ancien protocole (2020-21 à 2025-26, une saison testée par pli ; rapport préliminaire 1) reste
disponible en définissant `NHL_EXP=base` avant les imports ; ses sorties sont suffixées `_base`
(`tables_base/`, `figures_base/`, `experimentation_base/`).

In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from nhl_forecast.evaluation.preliminaire import (
    GRAINE,
    ACPKaiser,
    exporter_tableau,
    formater,
    plis_chronologiques,
    predictions,
    resultats_par_pli,
    resume,
    sauver_predictions,
    tableau_plis,
)
from nhl_forecast.features.equipes import (
    NOM_SAISON,
    charger_equipes,
    construire_matchs,
    variables,
)

SAUVER = True
NOM = "Régression logistique"

In [2]:
matchs = construire_matchs(charger_equipes())
X_cols = variables(matchs)
X, y = matchs[X_cols], matchs["victoire_locale"].to_numpy()
plis = plis_chronologiques(matchs)
print(f"{len(matchs)} matchs, {len(X_cols)} variables")

20382 matchs, 1060 variables


C:\Users\tom_9\Projets\nhl-probabilistic-forecasting\src\nhl_forecast\features\equipes.py:205: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  matchs = matchs.astype({"visiteur": "int64"}).reset_index()


## 1. Données en entrée

Une ligne par match. Pour chacune des 529 statistiques d'équipe (cinq sources), on calcule
pour chaque équipe, **avant** le match :

- `_saison` : la moyenne de ses matchs précédents de la saison ;
- `_10` : la moyenne de ses 10 derniers matchs (au moins 3, sinon manquant).

La variable du match est l'**écart local − visiteur** de ces moyennes (préfixe `d_`).
S'ajoutent l'écart de jours de repos et de matchs joués. Total : 529 × 2 + 2 = 1 060 variables. Le match lui-même n'entre jamais dans ses propres variables. Exemple pour
une statistique :

In [3]:
ex = [c for c in X_cols if c.startswith("d_mp_all__xGoalsPercentage")] + ["d_repos", "d_matchs_joues"]
formater(matchs[["date", "local_nom", "visiteur_nom", *ex, "victoire_locale"]].dropna().tail(5), 3)

date,local_nom,visiteur_nom,d_mp_all__xGoalsPercentage_saison,d_mp_all__xGoalsPercentage_10,d_repos,d_matchs_joues,victoire_locale
2026-04-16 00:00:00,Winnipeg Jets,San Jose Sharks,0.015,0.082,1.000,0,0
2026-04-16 00:00:00,Utah Mammoth,St. Louis Blues,0.032,-0.026,0.000,0,0
2026-04-16 00:00:00,Calgary Flames,Los Angeles Kings,-0.039,-0.128,0.000,0,1
2026-04-16 00:00:00,Edmonton Oilers,Vancouver Canucks,0.067,0.079,1.000,0,1
2026-04-16 00:00:00,Colorado Avalanche,Seattle Kraken,0.098,0.098,1.000,0,1


## 2. Plis chronologiques

Les 17 saisons 2009-10 à 2025-26 sont utilisées. Chaque pli teste une saison et
s'entraîne sur toutes les saisons antérieures (*walk-forward*). Les saisons 2009-10 à
2015-16 ne servent qu'à l'entraînement ; les plis 1 à 8 (2016-17 à 2023-24) servent au
développement, les plis 9 et 10 (2024-25 et 2025-26) à l'évaluation finale.

In [4]:
formater(tableau_plis(matchs))

Pli,Phase,Saisons d'entraînement,Saison testée,Matchs (entraînement),dont validation interne,Matchs (test)
1,Développement,2009-10 à 2015-16,2016-17,8100,1620,1230
2,Développement,2009-10 à 2016-17,2017-18,9330,1866,1271
3,Développement,2009-10 à 2017-18,2018-19,10601,2120,1271
4,Développement,2009-10 à 2018-19,2019-20,11872,2374,1082
5,Développement,2009-10 à 2019-20,2020-21,12954,2591,868
6,Développement,2009-10 à 2020-21,2021-22,13822,2764,1312
7,Développement,2009-10 à 2021-22,2022-23,15134,3027,1312
8,Développement,2009-10 à 2022-23,2023-24,16446,3289,1312
9,Évaluation finale,2009-10 à 2023-24,2024-25,17758,3552,1312
10,Évaluation finale,2009-10 à 2024-25,2025-26,19070,3814,1312


## 3. Entraînement et résultats par pli (saison testée)

In [5]:
def pipeline():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         ACPKaiser(random_state=GRAINE),
                         LogisticRegression(max_iter=5000))  # fmt: skip


preds, acp = [], []
for pli in plis:
    m = pipeline().fit(X.iloc[pli["entr"]], y[pli["entr"]])
    preds.append(predictions(matchs, pli, m.predict_proba(X.iloc[pli["test"]])[:, 1], NOM))
    acp.append({"Pli": pli["pli"], "Saison testée": NOM_SAISON[pli["saison_test"]],
                "Composantes ACP": m[-2].n_components_,
                "Variance expliquée (%)": round(100 * m[-2].variance_expliquee_, 1)})
preds = pd.concat(preds, ignore_index=True)
tab_acp = pd.DataFrame(acp)
tab_plis = resultats_par_pli(preds)
if SAUVER:
    exporter_tableau(tab_plis, "exp_reglog_par_pli")
    exporter_tableau(tab_acp, "exp_reglog_composantes")
    sauver_predictions(preds, "01_reglog")
formater(tab_acp)

Pli,Saison testée,Composantes ACP,Variance expliquée (%)
1,2016-17,152,90.6000
2,2017-18,153,90.4000
3,2018-19,155,90.3000
4,2019-20,156,90.2000
5,2020-21,157,90.2000
6,2021-22,158,90.2000
7,2022-23,159,90.2000
8,2023-24,158,90.1000
9,2024-25,159,90.1000
10,2025-26,159,90.0000


In [6]:
formater(tab_plis)

Modèle,Pli,Phase,Saison testée,n,Log loss,Brier,Accuracy
Régression logistique,1,Développement,2016-17,1230,0.6816,0.2443,0.5675
Régression logistique,2,Développement,2017-18,1271,0.6780,0.2423,0.5799
Régression logistique,3,Développement,2018-19,1271,0.6844,0.2453,0.5720
Régression logistique,4,Développement,2019-20,1082,0.6902,0.2483,0.5444
Régression logistique,5,Développement,2020-21,868,0.6857,0.2459,0.5553
Régression logistique,6,Développement,2021-22,1312,0.6609,0.2343,0.6212
Régression logistique,7,Développement,2022-23,1312,0.6705,0.2383,0.5892
Régression logistique,8,Développement,2023-24,1312,0.6729,0.2400,0.5953
Régression logistique,9,Évaluation finale,2024-25,1312,0.6682,0.2380,0.5877
Régression logistique,10,Évaluation finale,2025-26,1312,0.6858,0.2463,0.5351


## 4. Résumé (moyenne ± écart-type sur les saisons testées, par phase)

In [7]:
tab_resume = resume(preds)
if SAUVER:
    exporter_tableau(tab_resume, "exp_reglog_resume")
formater(tab_resume)

Modèle,Phase,Saisons,Log loss,Brier,Accuracy
Régression logistique,Développement,8,0.6780 ± 0.0095,0.2423 ± 0.0046,0.5781 ± 0.0242
Régression logistique,Évaluation finale,2,0.6770 ± 0.0124,0.2421 ± 0.0059,0.5614 ± 0.0372
